In [ ]:
from typing import Annotated, List, Any
from operator import add

from IPython.display import Image, display
from jinja2 import Template
from pydantic import BaseModel, Field
import cohere
import instructor
import openai

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, convert_to_openai_messages
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langsmith import traceable, get_current_run_tree
from qdrant_client import QdrantClient, models
from qdrant_client.models import Document, FieldCondition, Filter, FusionQuery, MatchAny, Prefetch, MatchValue
from langgraph.checkpoint.postgres import PostgresSaver
from langchain_core.messages import ToolMessage
import uuid

import psycopg2
from psycopg2.extras import RealDictCursor
import numpy as np

In [ ]:
## ads these items to the shopping cart .... 
def add_to_shopping_cart(items: list[dict], user_id: str, cart_id: str) -> str:
    """Add a list of provided items to the shopping cart.

    Args:
        items: A list of items to add to the shopping cart. Each item is a dictionary with the following keys: product_id, quantity.
        user_id: The id of the user to add the items to the shopping cart.
        cart_id: The id of the shopping cart to add the items to.

    Returns:
        A list of the items added to the shopping cart.
    """

    conn = psycopg2.connect(
        host="localhost",
        port=5433,
        database="tools_database",
        user="tools_user",
        password="tools_user_password"
    )

    conn.autocommit = True
    
    with conn.cursor(cursor_factory=RealDictCursor) as cursor:
        for item in items:
            print(item)
            product_id = item['product_id']
            quantity = item['quantity']

            qdrant_client = QdrantClient(url="http://localhost:6333")

            dummy_vector = np.zeros(1536).tolist()
            res = qdrant_client.query_points(
                collection_name="Amazon-items-collection-01-hybrid-search",
                prefetch=[
                    Prefetch(
                        query=dummy_vector,
                        filter=Filter(
                            must=[
                                FieldCondition(
                                    key="parent_asin",
                                    match=MatchValue(value=product_id)
                                )
                            ]
                        ),
                        using="text-embedding-3-small",
                        limit=20
                    )
                ],
                query=FusionQuery(fusion="rrf"),
                limit=1
            )
            print(res)
            payload = res.points[0].payload
            
            product_image_url = payload.get("image")
            price = payload.get("price")
            currency = "USD"
            
            # Check if item already exists
            check_query = """
                SELECT id, quantity, price
                FROM shopping_carts.shopping_cart_items
                WHERE user_id = %s AND shopping_cart_id = %s AND product_id = %s
            """
            cursor.execute(check_query, (user_id, cart_id, product_id))
            existing_item = cursor.fetchone() # fetch one item and if it exists

            if existing_item:
                # Update existing item
                new_quantity = existing_item['quantity'] + quantity

                update_query = """
                UPDATE shopping_carts.shopping_cart_items
                SET
                    quantity = %s,
                    price = %s,
                    currency = %s,
                    product_image_url = COALESCE(%s, product_image_url)
                WHERE user_id = %s AND shopping_cart_id = %s AND product_id = %s
                RETURNING id, quantity, price
                """
                cursor.execute(update_query, (new_quantity, price, currency, product_image_url, user_id, cart_id, product_id))
            else:
                # Insert new item
                insert_query = """
                    INSERT INTO shopping_carts.shopping_cart_items (
                        user_id, shopping_cart_id, product_id, quantity, price, currency, product_image_url)
                    VALUES (%s, %s, %s, %s, %s, %s, %s)
                    RETURNING id, quantity, price
                """
                cursor.execute(insert_query, (user_id, cart_id, product_id, quantity, price, currency, product_image_url))
    
    return f"Added {items} items to the shopping cart."

In [ ]:
items = [
    {
        "product_id": "B09MM5QT3R",
        "quantity": 2,
    },
    {
        "product_id": "B0BKPB2YQ9",
        "quantity": 4
    },
    
]
add_to_shopping_cart(items, "test_user_1", "test_cart_1")

### get the shopping cart items tool

In [ ]:
def get_shopping_cart(user_id: str, cart_id: str) -> list[dict]:
    """
    Retrieve all items in a user's shopping cart.

    Args:
        user_id: User identifier
        cart_id: Cart identifier

    Returns:
        list of dictionaries containing cart items
    """
    conn = psycopg2.connect(
        host="localhost",
        port=5433,
        database="tools_database",
        user="tools_user",
        password="tools_user_password"
    )
    conn.autocommit = True

    with conn.cursor(cursor_factory=RealDictCursor) as cursor:
        query = """
        SELECT
            product_id, price, quantity,
            currency, product_image_url,
            (price * quantity) as total_price
        FROM shopping_carts.shopping_cart_items
        WHERE user_id = %s AND shopping_cart_id = %s
        ORDER BY added_at DESC
        """  
        cursor.execute(query, (user_id, cart_id))
        
        return [row for row in cursor.fetchall()]

In [ ]:
get_shopping_cart("test_user_1", "test_cart_1")

### Deleting items from Shopping Cart Tool

In [ ]:
def remove_from_cart(product_id: str, user_id: str, cart_id: str) -> str:
    """
    Remove an item completely from the shopping cart.

    Args:
        user_id: User identifier
        product_id: Product identifier to remove
        cart_id: Cart identifier

    Returns:
        Information about the removal of the item from the shopping cart.
    """
    conn = psycopg2.connect(
        host="localhost",
        port=5433,
        database="tools_database",
        user="tools_user",
        password="tools_user_password"
    )
    conn.autocommit = True
    with conn.cursor(cursor_factory=RealDictCursor) as cursor:
        query = """
        DELETE FROM shopping_carts.shopping_cart_items
        WHERE user_id = %s AND shopping_cart_id = %s AND product_id = %s
        """
        cursor.execute(query, (user_id, cart_id, product_id))

        return f"Removed {product_id} from the shopping cart." if cursor.rowcount > 0 else f"Item {product_id} not found in the shopping cart."

In [ ]:
remove_from_cart("B09MM5QT3R", "test_user_1", "test_cart_1")

In [ ]:
get_shopping_cart("test_user_1", "test_cart_1")

In [ ]:
remove_from_cart("B0BKPB2YQ9", "test_user_1", "test_cart_1")

In [ ]:
get_shopping_cart("test_user_1", "test_cart_1")